<a href="https://colab.research.google.com/github/PravalikaGangoni/Financial-RAG-Chatbot/blob/main/financial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Financial RAG Chatbot for Google Colab
# ============================================================
#
# Beginner-friendly project:
# - Upload PDF, TXT, and CSV financial documents
# - Read and clean document text
# - Split text into chunks
# - Create embeddings using Sentence Transformers
# - Store embeddings in FAISS
# - Ask questions using Groq llama3-8b-8192
# - Chat with a Gradio interface
# - Show source text used for answers
# - Download chat history
#
# HOW TO USE IN GOOGLE COLAB:
# 1. Open https://colab.research.google.com/
# 2. Create a new notebook
# 3. Paste this full file into one Colab cell
# 4. Run the cell
# 5. Paste your Groq API key when Colab asks for it
# ============================================================


# ============================================================
# STEP 1: Install All Required Libraries
# ============================================================

# These commands install the free Python libraries used in this project.
# We use normal Python installation code so this file works as a .py file too.
import subprocess
import sys

# Install LangChain, Groq support, and HuggingFace support.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "langchain",
    "langchain-community",
    "langchain-groq",
    "langchain-huggingface",
])

# Install embeddings, vector database, UI, PDF, and CSV tools.
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "sentence-transformers",
    "faiss-cpu",
    "gradio",
    "pypdf",
    "pandas",
])


# ============================================================
# STEP 2: Import Libraries
# ============================================================

# os helps us work with files and folders.
import os

# getpass asks for the API key without showing it on screen.
from getpass import getpass

# re helps us clean text using simple patterns.
import re

# pandas helps us read CSV files.
import pandas as pd

# gradio helps us build the chatbot web interface.
import gradio as gr

# PdfReader helps us extract text from PDF files.
from pypdf import PdfReader

# Document is LangChain's standard format for storing text plus metadata.
# Newer LangChain versions use langchain_core.documents.
try:
    from langchain_core.documents import Document
except ImportError:
    from pydantic import BaseModel
    class Document(BaseModel):
        page_content: str
        metadata: dict

# This splitter breaks long text into smaller chunks.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# This creates free embeddings using Sentence Transformers.
from langchain_huggingface import HuggingFaceEmbeddings

# FAISS stores embeddings and allows fast similarity search.
from langchain_community.vectorstores import FAISS

# ChatGroq connects LangChain to Groq's hosted LLM API.
from langchain_groq import ChatGroq


# ============================================================
# STEP 3: Add Your Groq API Key
# ============================================================

# IMPORTANT:
# Do not paste your API key directly into the code.
# This hidden input box is safer and avoids showing your key on screen.
# You can get a Groq API key from https://console.groq.com/keys
GROQ_API_KEY = getpass("Paste your Groq API key here: ")


# ============================================================
# STEP 4: Project Settings
# ============================================================

# This folder stores the saved FAISS vector database.
FAISS_FOLDER = "financial_faiss_index"

# This list stores chat history for download.
chat_history_list = []

# This global variable will hold the final RAG chain after documents are processed.
qa_chain = None


# ============================================================
# STEP 5: Read PDF Files
# ============================================================

def read_pdf(file_path):
    """
    Read text from a PDF file.

    Args:
        file_path: Path to the PDF file.

    Returns:
        Extracted text as a string.
    """

    # Start with empty text.
    text = ""

    try:
        # Open the PDF file.
        pdf_reader = PdfReader(file_path)

        # Read each page one by one.
        for page in pdf_reader.pages:

            # Extract text from the current page.
            page_text = page.extract_text()

            # Add page text only if text was found.
            if page_text:
                text += page_text + "\n"

    except Exception as error:
        # Print a friendly error if the PDF cannot be read.
        print(f"Error reading PDF {file_path}: {error} - financial_rag_chatbot_colab.py:166")

    return text


# ============================================================
# STEP 6: Read TXT Files
# ============================================================

def read_txt(file_path):
    """
    Read text from a TXT file.

    Args:
        file_path: Path to the TXT file.

    Returns:
        File text as a string.
    """

    try:
        # Open the text file using UTF-8.
        with open(file_path, "r", encoding="utf-8", errors="ignore") as file:
            return file.read()

    except Exception as error:
        # Print a friendly error if the TXT file cannot be read.
        print(f"Error reading TXT {file_path}: {error} - financial_rag_chatbot_colab.py:193")
        return ""


# ============================================================
# STEP 7: Read CSV Files
# ============================================================

def read_csv(file_path):
    """
    Read text from a CSV file.

    Args:
        file_path: Path to the CSV file.

    Returns:
        CSV contents converted to readable text.
    """

    try:
        # Read the CSV into a pandas DataFrame.
        dataframe = pd.read_csv(file_path)

        # Convert the table into plain text.
        return dataframe.to_string(index=False)

    except Exception as error:
        # Print a friendly error if the CSV cannot be read.
        print(f"Error reading CSV {file_path}: {error} - financial_rag_chatbot_colab.py:221")
        return ""


# ============================================================
# STEP 8: Read Uploaded Financial Documents
# ============================================================

def read_financial_documents(file_paths):
    """
    Read PDF, TXT, and CSV files and convert them into LangChain Documents.

    Args:
        file_paths: List of uploaded file paths.

    Returns:
        List of LangChain Document objects.
    """

    # This list will store all readable documents.
    documents = []

    # Go through every uploaded file.
    for file_path in file_paths:

        # Get only the file name, not the full path.
        file_name = os.path.basename(file_path)

        # Get the file extension, such as pdf, txt, or csv.
        file_extension = file_name.lower().split(".")[-1]

        # Read PDF files.
        if file_extension == "pdf":
            text = read_pdf(file_path)

        # Read TXT files.
        elif file_extension == "txt":
            text = read_txt(file_path)

        # Read CSV files.
        elif file_extension == "csv":
            text = read_csv(file_path)

        # Skip unsupported files.
        else:
            print(f"Unsupported file type: {file_name} - financial_rag_chatbot_colab.py:266")
            continue

        # Store the text if the file had readable content.
        if text.strip():
            documents.append(
                Document(
                    page_content=text,
                    metadata={"source": file_name}
                )
            )

    return documents


# ============================================================
# STEP 9: Clean And Preprocess Text
# ============================================================

def clean_text(text):
    """
    Clean text by removing extra spaces and line breaks.

    Args:
        text: Raw text from a document.

    Returns:
        Cleaned text.
    """

    # Replace new lines with spaces.
    text = text.replace("\n", " ")

    # Replace many spaces with one space.
    text = re.sub(r"\s+", " ", text)

    # Remove spaces from the beginning and end.
    text = text.strip()

    return text


def clean_documents(documents):
    """
    Clean every LangChain Document.

    Args:
        documents: List of raw LangChain Documents.

    Returns:
        List of cleaned LangChain Documents.
    """

    # This list will store cleaned documents.
    cleaned_documents = []

    # Go through each document.
    for document in documents:

        # Clean the document text.
        cleaned_content = clean_text(document.page_content)

        # Keep the document only if it still has text.
        if cleaned_content:
            cleaned_documents.append(
                Document(
                    page_content=cleaned_content,
                    metadata=document.metadata
                )
            )

    return cleaned_documents


# ============================================================
# STEP 10: Split Text Into Chunks
# ============================================================

def split_documents_into_chunks(documents):
    """
    Split long documents into smaller chunks for better retrieval.

    Args:
        documents: List of cleaned LangChain Documents.

    Returns:
        List of smaller Document chunks.
    """

    # Create a text splitter.
    text_splitter = RecursiveCharacterTextSplitter(

        # Each chunk will be about 1000 characters.
        chunk_size=1000,

        # Chunks overlap slightly so context is not lost.
        chunk_overlap=150
    )

    # Split documents into chunks.
    chunks = text_splitter.split_documents(documents)

    # Show how many chunks were created.
    print(f"Created {len(chunks)} text chunks. - financial_rag_chatbot_colab.py:369")

    return chunks


# ============================================================
# STEP 11: Create Embeddings
# ============================================================

def create_embedding_model():
    """
    Create a free Sentence Transformers embedding model.

    Returns:
        HuggingFaceEmbeddings model.
    """

    # This model is small, free, and works well for beginner RAG projects.
    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embedding_model


# ============================================================
# STEP 12: Create And Save FAISS Vector Database
# ============================================================

def create_and_save_faiss_database(chunks, embedding_model):
    """
    Create and save a FAISS vector database.

    Args:
        chunks: List of document chunks.
        embedding_model: Sentence Transformers embedding model.

    Returns:
        FAISS vector database.
    """

    # Stop early if no chunks were created.
    if len(chunks) == 0:
        raise ValueError("No text chunks found. Please upload valid documents.")

    # Create the FAISS vector database from chunks.
    vector_database = FAISS.from_documents(
        documents=chunks,
        embedding=embedding_model
    )

    # Save FAISS locally inside Colab.
    vector_database.save_local(FAISS_FOLDER)

    print("FAISS database saved successfully. - financial_rag_chatbot_colab.py:423")

    return vector_database


# ============================================================
# STEP 13: Connect Groq API
# ============================================================

def create_groq_llm():
    """
    Connect to Groq LLM using LangChain.

    Returns:
        Groq chat model.
    """

    # Check if the beginner left the API key box empty.
    if not GROQ_API_KEY.strip():
        raise ValueError("Please paste your real Groq API key first.")

    # Create the Groq LLM.
    llm = ChatGroq(

        # Your Groq API key.
        groq_api_key=GROQ_API_KEY,

        # Required model for this project.
        model_name="llama-3.1-8b-instant",

        # Lower temperature gives more focused financial answers.
        temperature=0.2
    )

    return llm


# ============================================================
# STEP 14: Create Simple RAG Chain
# ============================================================

class SimpleFinancialRAGChain:
    """
    A beginner-friendly RAG chain.

    This replaces LangChain's older RetrievalQA import, because that import
    can fail in newer Colab/LangChain versions.
    """

    def __init__(self, vector_database, llm):
        # Store the FAISS database.
        self.vector_database = vector_database

        # Store the Groq LLM.
        self.llm = llm

    def invoke(self, inputs):
        """
        Answer one question using retrieved document chunks.

        Args:
            inputs: Dictionary with a "query" key.

        Returns:
            Dictionary with "result" and "source_documents".
        """

        # Get the question from the input dictionary.
        question = inputs["query"]

        # Search FAISS for the top 4 most relevant chunks.
        source_documents = self.vector_database.similarity_search(question, k=4)

        # Combine retrieved chunks into one context string.
        context = "\n\n".join(
            document.page_content for document in source_documents
        )

        # Create a clear financial prompt for the LLM.
        prompt = f"""
You are a helpful financial document assistant.

Use only the provided context to answer the question.
If the answer is not found in the context, say:
"I could not find this information in the uploaded documents."

Format financial answers clearly.
Use bullet points when helpful.
Do not invent numbers.

Context:
{context}

Question:
{question}

Answer:
"""

        # Send the prompt to Groq.
        response = self.llm.invoke(prompt)

        # ChatGroq usually returns an object with a .content field.
        if hasattr(response, "content"):
            answer = response.content
        else:
            answer = str(response)

        # Return the same format the chatbot function expects.
        return {
            "result": answer,
            "source_documents": source_documents
        }


def create_retrieval_qa_chain(vector_database, llm):
    """
    Create the RAG question-answering chain.

    Args:
        vector_database: FAISS vector database.
        llm: Groq language model.

    Returns:
        SimpleFinancialRAGChain object.
    """

    # Return our simple beginner-friendly RAG chain.
    return SimpleFinancialRAGChain(vector_database, llm)


# ============================================================
# STEP 15: Process Uploaded Files From Gradio
# ============================================================

def process_gradio_files(uploaded_files):
    """
    Process files uploaded in the Gradio interface.

    Args:
        uploaded_files: Files uploaded by the user.

    Returns:
        Status message for the user.
    """

    # Use the global qa_chain so the chatbot can access it later.
    global qa_chain

    # Make sure the user uploaded at least one file.
    if not uploaded_files:
        return "Please upload at least one PDF, TXT, or CSV file."

    try:
        # Get the path of each uploaded file.
        file_paths = [file.name for file in uploaded_files]

        # Read uploaded documents.
        raw_documents = read_financial_documents(file_paths)

        # Clean the document text.
        cleaned_docs = clean_documents(raw_documents)

        # Split documents into chunks.
        chunks = split_documents_into_chunks(cleaned_docs)

        # Create the embedding model.
        embedding_model = create_embedding_model()

        # Create and save the FAISS vector database.
        vector_database = create_and_save_faiss_database(chunks, embedding_model)

        # Connect to Groq.
        llm = create_groq_llm()

        # Create the full RAG chain.
        qa_chain = create_retrieval_qa_chain(vector_database, llm)

        # Return a success message.
        return f"Documents processed successfully. Created {len(chunks)} chunks."

    except Exception as error:
        # Return a friendly error message.
        return f"Error while processing documents: {error}"


# ============================================================
# STEP 16: Chatbot Function
# ============================================================

def ask_financial_chatbot(question, chat_history):
    """
    Answer a user question using the RAG chain.

    Args:
        question: User question.
        chat_history: Existing Gradio chat history.

    Returns:
        Updated chat history and source text.
    """

    # Use the global RAG chain created after processing documents.
    global qa_chain

    # Make sure documents were processed first.
    if qa_chain is None:
        return chat_history, "Please upload and process documents first."

    # Make sure the user typed a real question.
    if not question.strip():
        return chat_history, "Please type a question."

    try:
        # Ask the RAG chain the question.
        result = qa_chain.invoke({"query": question})

        # Extract the generated answer.
        answer = result["result"]

        # Extract the source documents used to answer.
        source_documents = result["source_documents"]

        # Prepare readable source text for the user.
        source_text = ""

        # Show a short preview of each retrieved source.
        for index, document in enumerate(source_documents, start=1):

            # Get the original file name.
            source_name = document.metadata.get("source", "Unknown source")

            # Show only a preview so the box does not get too long.
            preview = document.page_content[:700]

            # Add source information to the output.
            source_text += f"\nSOURCE {index}: {source_name}\n"
            source_text += preview + "\n"

        # Add the conversation to Gradio chat history.
        chat_history.append((question, answer))

        # Add the conversation to downloadable history.
        chat_history_list.append(
            {
                "question": question,
                "answer": answer
            }
        )

        return chat_history, source_text

    except Exception as error:
        # Show a friendly error inside the chat.
        error_message = f"Something went wrong: {error}"

        # Add the error to chat history.
        chat_history.append((question, error_message))

        return chat_history, ""


# ============================================================
# STEP 17: Download Chat History
# ============================================================

def download_chat_history():
    """
    Save chat history to a text file and return the file path.

    Returns:
        File path for Gradio download.
    """

    # Name of the chat history file.
    file_name = "financial_chat_history.txt"

    # Write chat history to the file.
    with open(file_name, "w", encoding="utf-8") as file:

        # Write each question and answer.
        for item in chat_history_list:
            file.write(f"Question: {item['question']}\n")
            file.write(f"Answer: {item['answer']}\n")
            file.write("-" * 60 + "\n")

    return file_name


# ============================================================
# STEP 18: Clear Chat
# ============================================================

def clear_chat():
    """
    Clear the visible chat and source box.

    Returns:
        Empty chat history and empty source text.
    """

    return [], ""


# ============================================================
# STEP 19: Create Gradio User Interface
# ============================================================

# Blocks lets us build a clean custom interface.
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    # Main title.
    gr.Markdown("# Financial RAG Chatbot")

    # Beginner-friendly instruction.
    gr.Markdown(
        "Upload financial PDF, TXT, or CSV files. Then click **Process Documents** and ask questions."
    )

    # File upload section.
    file_input = gr.File(
        label="Upload Financial Documents",
        file_count="multiple",
        file_types=[".pdf", ".txt", ".csv"]
    )

    # Button to process uploaded documents.
    process_button = gr.Button("Process Documents")

    # Textbox to show processing messages.
    process_output = gr.Textbox(
        label="Processing Status",
        lines=2
    )

    # Chatbot display area.
    chatbot = gr.Chatbot(
        label="Financial Chatbot",
        height=350
    )

    # User question box.
    question_input = gr.Textbox(
        label="Ask a Question",
        placeholder="Example: What was the total revenue?",
        lines=2
    )

    # Buttons for submitting and clearing.
    with gr.Row():
        submit_button = gr.Button("Submit Question")
        clear_button = gr.Button("Clear Chat")

    # Source text box.
    source_output = gr.Textbox(
        label="Retrieved Source Text Used For Answering",
        lines=12
    )

    # Chat history download button.
    download_button = gr.Button("Download Chat History")

    # Downloadable file output.
    download_file = gr.File(label="Chat History File")

    # When the process button is clicked, process uploaded documents.
    process_button.click(
        fn=process_gradio_files,
        inputs=file_input,
        outputs=process_output
    )

    # When the submit button is clicked, answer the question.
    submit_button.click(
        fn=ask_financial_chatbot,
        inputs=[question_input, chatbot],
        outputs=[chatbot, source_output]
    )

    # Pressing Enter in the question box also submits the question.
    question_input.submit(
        fn=ask_financial_chatbot,
        inputs=[question_input, chatbot],
        outputs=[chatbot, source_output]
    )

    # Clear chat button.
    clear_button.click(
        fn=clear_chat,
        inputs=None,
        outputs=[chatbot, source_output]
    )

    # Download chat history button.
    download_button.click(
        fn=download_chat_history,
        inputs=None,
        outputs=download_file
    )


# ============================================================
# STEP 20: Launch The Chatbot In Google Colab
# ============================================================

# share=True creates a public temporary link, which is useful in Colab.
# debug=True shows helpful error messages while learning.
demo.launch(share=True, debug=True)


# ============================================================
# QUICK TESTING IDEAS
# ============================================================
#
# After launching:
# 1. Upload a financial PDF, TXT, or CSV file.
# 2. Click "Process Documents".
# 3. Ask questions like:
#    - What is the total revenue?
#    - What are the main expenses?
#    - Did net income increase or decrease?
#    - What financial risks are mentioned?
#    - Summarize the company performance.
#
# COMMON BEGINNER ERRORS:
# - If you see "Please paste your real Groq API key first",
#   replace the GROQ_API_KEY placeholder near the top of this file.
# - If you see "No text chunks found",
#   your document may be empty, scanned, or not readable as text.
# - If imports fail, run the cell again so pip installs finish correctly.
# - If Groq fails, check your API key and Groq usage limits.
#
# SIMPLE RAG EXPLANATION:
# RAG means Retrieval-Augmented Generation.
# First, the app searches your uploaded documents for relevant text.
# Then, it gives that text to the AI model so the answer is based on your files.
#
# FUTURE IMPROVEMENTS:
# - Add OCR for scanned PDFs.
# - Add Excel file support.
# - Add page numbers for citations.
# - Add charts for revenue and profit.
# - Store multiple document collections.
# ============================================================


/tmp/ipykernel_10848/3262927775.py:101: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Paste your Groq API key here: ··········


/tmp/ipykernel_10848/3262927775.py:732: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_10848/3262927775.py:759: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
/tmp/ipykernel_10848/3262927775.py:759: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://711e500400576840e0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Created 2 text chunks. - financial_rag_chatbot_colab.py:369


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS database saved successfully. - financial_rag_chatbot_colab.py:423
